In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("dataset.csv")

# Step 1 
## drop duplicates

In [ ]:
df.drop_duplicates(inplace = True)
df.reset_index(inplace = True  , drop = True)

In [ ]:
df.shape

# Step 2
## drop columns = ['title','undertitle','date','propertydetail']

In [ ]:
df.drop(columns = ['title','undertitle','date','propertydetail'],inplace = True)

In [ ]:
df.head()

# Step 3
## cleaning price column

In [ ]:
# (i) deleteing punctions and trash
# df['price'] = df['price'].str.split("\r").str[-1].str.strip().str.split("L").str[0].str.replace(",",'').str.strip()

In [ ]:
# df['price'] = df['price'].str.split("\r").str[-1].str.strip().str.split("L").str[0].str.replace(",","").str.strip()

In [ ]:
df['price'] = df['price'].str.replace(r"[^0-9.]",'',regex = True)

In [ ]:
df['price'] = df['price'].apply(lambda x:x if len(str(x))!= 0 else np.nan).astype(float)

In [ ]:
df['price'].dtype

In [ ]:
df.head()

# Step 4
## Extracting address

In [ ]:
df['state'] = df['address'].str.split(",").str[-1].str.strip().str.lower()

In [ ]:
df['city'] = df['address'].str.split(",").str[0].str.strip().str.lower()

In [ ]:
df.shape

In [ ]:
df = df[df['state'] =='yangon']
df.reset_index(drop = True , inplace = True)

In [ ]:
df.drop(columns = ['address'] , inplace = True)

In [ ]:
df.head()

In [ ]:
df.shape

# Step 5
## converting key_value to Dataframe

In [ ]:
df['key_value'] = df['key_value'].str.replace("[","").str.replace("]","")

In [ ]:
df['key_value']

In [ ]:
data = []

for i in df['key_value'].str.split("),",regex = False):
    key = []
    value =[]
    for j in i:
        k = j.split(",")[0].replace("('",'').replace("'",'').strip()
        val = j.split(",")[1].replace("'",'').replace('"','').strip()
        key.append(k)
        value.append(val)
    data.append(dict(zip(key,value)))
key_value_df = pd.DataFrame(data)
    
        
key_value_df_back = key_value_df.copy()

In [ ]:
key_value_df.reset_index(inplace = True , drop = True)

In [ ]:
key_value_df.shape

In [ ]:
df.shape

In [ ]:
df.tail(3)

In [ ]:
key_value_df.tail(3)

In [ ]:
key_value_df.head()

# Step 6
## Select columns from key_value_df ['Type','Area','Bed Room','Bath Room','Floor']

In [ ]:
key_value_df.head(2)

In [ ]:
# key_value_df.drop(columns = ['Ownership','Furnishing Type','Phone',
#                              'Availability Date','Aircon conditioner details',
#                             'Room Partition Type',
#                             'Flooring Type',])

In [ ]:
df.tail(5)

In [ ]:
key_value_df.tail(5)

In [ ]:
key_value_df = key_value_df[['Type','Area','Bed Room','Bath Room','Floor']]
key_value_df.tail(5)

# Step 7
## Select Only ['Apartment','House','Condo'] from Type Variable

In [ ]:
key_value_df['Type'].value_counts()

In [ ]:
key_value_df.shape

In [ ]:
key_value_df = key_value_df[key_value_df['Type'].isin(['Apartment','House','Condo'])]

In [ ]:
key_value_df.shape

In [ ]:
key_value_df.head()

# Step 8
## Cleaning Area Column

In [ ]:
# Foot square_1 from foot square
key_value_df['ft_square_1'] = key_value_df['Area'].apply(lambda x:x if 'ft<sup>2</sup>' in str(x) else str(0))
key_value_df['ft_square_1'] = key_value_df['ft_square_1'].str.replace("ft<sup>2</sup>","").astype(float)

In [ ]:
# Foot square_2 from x_and_y
key_value_df['area_x_y'] = key_value_df['Area'].apply(lambda x:x if 'x' in str(x) else str(0))
x = key_value_df['area_x_y'].str.split("x").str[0].astype(float)
y = key_value_df['area_x_y'].str.split("x").str[-1].astype(float)
key_value_df['ft_square_2'] = x*y

In [ ]:
# Foot square_3 from acre
# 1 acre = 43560 foot square
key_value_df['acre'] = key_value_df['Area'].apply(lambda x :x if "acre" in str(x) else str(0))
key_value_df['acre'] = key_value_df['acre'].str.replace("acre","").astype(float)
key_value_df['ft_square_3'] = key_value_df['acre'] * 43560

In [ ]:
# Adding at one column the Foot square
key_value_df["ft_square"] = key_value_df['ft_square_1'] + key_value_df['ft_square_2'] + key_value_df['ft_square_3'].isnull().sum()

In [ ]:
key_value_df.head()

In [ ]:
key_value_df.drop(columns = ['Area','area_x_y','acre','ft_square_1','ft_square_2','ft_square_3'] , inplace = True)

In [ ]:
key_value_df.shape

In [ ]:
# removing the the row where ft_square is zero
key_value_df = key_value_df[key_value_df['ft_square'] != 0.0]

In [ ]:
key_value_df.shape

# Step 9
## Cleaning Bed Room

In [ ]:
# converting the data type
key_value_df['Bed Room'] = key_value_df['Bed Room'].astype(float)

# Step 10
## Cleaning Bath Room

In [ ]:
# converting the data type 
key_value_df['Bath Room']=key_value_df['Bath Room'].astype(float)

# Step 11
## Cleaning Floor

In [ ]:
from floor_transcription import floor_trans

In [ ]:
def floor_str_to_int(x):
    key = list(floor_trans.keys())
    value = list(floor_trans.values())
    try:
        if x in key:
            ind = key.index(x)
            floor = value[ind]
            return float(np.round(floor,1))
        elif float(x):
            return x
    except:
        pass
    
    x = str(x).strip()
    if x ==str(np.nan) or x =="":
        return float(np.nan)
        


In [ ]:
key_value_df['Floor'] = key_value_df['Floor'].apply(floor_str_to_int).astype(float)

In [ ]:
key_value_df['Floor'].isnull().sum()

In [ ]:
# key_value_df = key_value_df.dropna()

In [ ]:
key_value_df.isnull().sum()

In [ ]:
key_value_df_back.shape

In [ ]:
result = pd.concat([df , key_value_df] , axis = 1)

In [ ]:
key_value_df

In [ ]:
result

In [ ]:
result.sample(10)

In [ ]:
# Step 12
# Delete key_value and apply dropna on the whole dataset

In [ ]:
result.drop(columns = ['key_value','state'] , inplace = True)

In [ ]:
result.dropna(inplace = True)

In [ ]:
result.shape

In [ ]:
# And the save the dataset

In [ ]:
result.info()

In [ ]:
result['city'].unique()

In [ ]:
result['Type'].unique()

In [ ]:
result

In [ ]:
result.select_dtypes(exclude='O').columns

In [ ]:
for i in result.select_dtypes(exclude = "O").columns:
    print(f"- {i}")

In [ ]:
import yaml

In [ ]:
with open(r"C:\Users\M S I\Desktop\Myanmar_Houses_Price_Prediction\data_schema\schema.yaml",'r') as file:
    yml_file = yaml.safe_load(file)

In [ ]:
col = yml_file['full_columns'].sort()

In [ ]:
col

In [ ]:
lst = list (result.columns)

In [ ]:
lst

In [ ]:
lst

In [ ]:
lst == yml_file['full_columns']

In [ ]:
yml_file['cat_columns']

# Data Feature Engineering

In [ ]:
result['price']

In [ ]:
result['city'].value_counts()

In [ ]:
result['Type'].value_counts()

In [ ]:
int(result['city'].value_counts()['hlaing'])

In [ ]:
def arrange_cat(series, threshold=20):
    # 1. Calculate counts once for the whole series
    counts = series.value_counts()
    
    # 2. Identify categories to keep
    keep = counts[counts >= threshold].index
    
    # 3. Use .where() to replace anything not in 'keep' with 'Others'
    return series.where(series.isin(keep), 'Others')

# Usage:
# df['city'] = arrange_cat(df['city'])

In [ ]:
counts = result['city'].value_counts()

In [ ]:
result['city'].mask(~result['city'].isin(counts[counts>=20].index),'Others')

In [ ]:
# The "Pro" way (Vectorized)
result['city'].mask(~result['city'].isin(counts[counts >= 20].index), 'Others').value_counts()

In [ ]:
result['price'].quantile(0.25)

In [ ]:
def outlier_treatment(x):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    if x < lower:
        return lower
    elif x>=upper:
        return upper

In [ ]:
def outlier_treatment(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    
    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)
    
    # Efficiently caps values below lower and above upper
    return series.clip(lower=lower, upper=upper)

# Usage
result['price'] = outlier_treatment(result['price'])

In [ ]:
def arrange_cat(x):
    count = x.value_counts()
    rare_count = count[count<20].index
    return x.replace(rare_count , 'Others')

In [ ]:
arrange_cat(result['city']).value_counts()

In [ ]:
outlier_treatment(result['ft_square']).plot(kind = 'box')

In [ ]:
result.columns

In [ ]:
result['price'].describe()

In [ ]:
dct = {'Train':[{'Column':{'price':[{'count':4 , 'mean':34.4,"median":4.2}],'Bed Room':[1,2,3,4],"Bath Room":[1,2,3,4,5]}}]}

In [ ]:
with open('example.yaml','w') as file:
    yaml.safe_dump(final , file)

In [ ]:
result['price'].dtype

In [ ]:
price_dict = dict(zip(result['price'].describe().index,result['price'].describe()))

In [ ]:
dict(zip(result['Type'].value_counts().index , result['Type'].value_counts()))

In [ ]:
column = {}
for i in result.columns:
    if result[i].dtype == 'O' or len(list(result[i].unique()))<=20:
        ind = result[i].value_counts().index
        val = list(result[i].value_counts())
        temp_dict = dict(zip(ind,list(val)))
        column[i]=(temp_dict)
    else:
        ind = result[i].describe().index
        val = list(result[i].describe())
        temp_dict = dict(zip(ind,list(val)))
        column[i]=(temp_dict)

final_column = {'Detail':[{"Train":[{'Columns':[column]}],'Test':[{'Columns':[column]}]}]}

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train , test = train_test_split(result , test_size = 0.2)

In [ ]:
def detail_for_yaml(train , test):
    final = {}
    df = {'Train':train , "Test":test}
    for text , data in list(zip(list(df.keys()) , list(df.values()))):
        column = {}
        for i in data.columns:
            if data[i].dtype == 'O' or len(list(data[i].unique()))<=20:
                ind = data[i].value_counts().index
                val = list(data[i].value_counts())
                temp_dict = dict(zip(ind,list(val)))
                column[i]=(temp_dict)
            else:
                ind = data[i].describe().index
                val = list(data[i].describe())
                temp_dict = dict(zip(ind,list(val)))
                column[i]=(temp_dict)
        final[text] = column
    return final

In [ ]:
detail_for_yaml(train , test)

In [ ]:
sorted(result['Floor'].value_counts().index)

In [ ]:
result['Floor'].value_counts().plot(kind = 'bar')

In [ ]:
np.round(result['Floor'],0).value_counts().plot(kind = 'bar')

In [ ]:
result['Floor'].apply(lambda x: x if x<6  else 7).unique()

In [ ]:
result['Floor']

In [ ]:
result

In [ ]:
np.array(result['price'])

In [ ]:
list(result['ft_square'].describe())

In [ ]:
result.rename(columns = {'Type':'type' , 'Bed Room':'bed_room' , 'Bath Room':'bath_room' , 'Floor':'floor'})